
# C7-cnn-transfer — Practice p13

**Type:** proof · **Difficulty:** core · **Concepts:** receptive field

**Reasoning is required.**

Derive the receptive-field growth formula from the sliding-window
definition itself.

Setup: a stack of 1-D convolutions; layer $\ell$ has kernel size
$K_\ell$ and stride $s_\ell$.
Define the **jump** $J_\ell = \prod_{i \le \ell} s_i$ (with
$J_0 = 1$): the distance in *input* pixels between two adjacent
positions of layer $\ell$'s output map.
Let $r_\ell$ be the receptive-field size (in input pixels) of one
output of layer $\ell$, with $r_0 = 1$.
**Standing assumption (holds for every architecture in this course,
including ResNet-50):** $J_{\ell-1} \le r_{\ell-1}$ — each step of the
previous map is no larger than what one of its outputs already sees, so
adjacent spans overlap or tile with no gaps. Your proof may (and must)
invoke it where needed.

**(a)** Prove the update
$$r_\ell = r_{\ell - 1} + (K_\ell - 1)\, J_{\ell - 1},$$
arguing from what one layer-$\ell$ output reads: $K_\ell$ positions
of the previous map, adjacent ones $J_{\ell-1}$ input pixels apart,
each seeing $r_{\ell-1}$ input pixels.
(State why the spans of adjacent positions overlap or tile so that
the union is an interval of exactly that length, and why $J$ updates
*after* $r$ at a striding layer.)

**(b)** Conclude the closed form
$$r_L = 1 + \sum_{\ell=1}^{L} (K_\ell - 1) \prod_{i < \ell} s_i,$$
and derive the stride-1 corollary $r_L = 1 + \sum (K_\ell - 1)$.

**(c) Concrete anchor.**
For the stack $K = (3, 5, 3)$, $s = (1, 2, 1)$: compute
`rf_formula` by hand from (b) (show each term), then measure
`rf_measured` empirically with the given all-ones `nn.Conv1d` stack:
for the center output index, count how many single-input
perturbations change it (the probe you write below).
`anchor_gap = abs(rf_formula - rf_measured)` must be `0`.

Your derivation in (a)–(b) must stand on its own — it may not appeal
to code output; the probe in (c) is a check, not an argument.



*Your derivation for (a) and (b) here.*


In [ ]:

import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention (no pretrained weights here)
SEED = 20260804


def ones_conv1d(K, s):
    c = nn.Conv1d(1, 1, kernel_size=K, stride=s, bias=False)
    c.weight = nn.Parameter(torch.ones(1, 1, K), requires_grad=False)
    return c


stack = nn.Sequential(ones_conv1d(3, 1), ones_conv1d(5, 2), ones_conv1d(3, 1))

N = 101                                   # long enough that the center is interior
base = torch.zeros(1, 1, N)
with torch.inference_mode():
    out0 = stack(base)
center = out0.shape[-1] // 2

# (c) YOUR CODE HERE:
rf_formula = ...   # plain int, from your closed form (show the terms in a comment)

# probe: perturb each input position; count how many change out[center]
rf_measured = ...  # YOUR CODE HERE (loop allowed here)
anchor_gap = ...   # YOUR CODE HERE -- must be 0
